# Run inference for EarTTS

In [ ]:
from vllm import SamplingParams
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.v1.engine.async_llm import AsyncLLM
from vllm.model_executor.models.eartts import EarTTSInputEmbedding

import torch
from omegaconf import OmegaConf

# audio tokens for prompt to derive speaker identity
prompt_audio_codes = torch.load("eartts_debug_tokens/code.pt").cuda()
prompt_audio_codes = torch.nn.functional.pad(prompt_audio_codes[0, :-1, :], (0, 0, 1, 0)).transpose(0, 1)  # T x 31
# subword ids corresponding to the text to synthesize
next_subword_ids = torch.load("eartts_debug_tokens/next_subword_ids.pt").cuda()  # T
# subword ids corresponding to the prompt, instruction, etc.
subword_ids_prompt = torch.load("eartts_debug_tokens/input_text_tokens.pt").cuda()
last_prompt_subword_id = subword_ids_prompt[:, -1]  # 1
subword_ids_prompt = subword_ids_prompt[0, :-1]  # T


# load vllm engine
type_str = "float16"
torch_type = getattr(torch, type_str)
engine_args = AsyncEngineArgs(
    model="eartts_vllm_model",
    dtype=type_str,
    max_model_len=256,
    gpu_memory_utilization=0.6,
    skip_tokenizer_init=True,  # Skip tokenizer since we're using embeddings directly
)
engine = AsyncLLM.from_engine_args(engine_args)
sampling_params = SamplingParams(max_tokens=64, skip_sampling=True)


# load embedding model that we use to create hidden states for vllm inference
conf = OmegaConf.load("eartts_vllm_model/eartts_input_embedding_config.yaml")
conf.model_dir = "eartts_vllm_model"
embedding = EarTTSInputEmbedding(conf)
embedding.load_state_dict(torch.load("eartts_vllm_model/eartts_input_embedding.ckpt"))
embedding.cuda()
embedding.to(torch_type)


# embeddings for context phase
prompt_embeds = embedding(
    context_text_tokens=subword_ids_prompt,
    audio_tokens=prompt_audio_codes,
).squeeze(0).detach().cpu().to(torch_type)
inputs = {
    # dummy tokens
    "prompt_token_ids": [0] * prompt_embeds.shape[0],
    # actual embeddings
    "custom_inputs": {"total_embeddings": prompt_embeds}
}
acoustic_tokens_lst = []
i = 0
context_subword_id = last_prompt_subword_id  # (1,)
async for output in engine.generate(inputs, sampling_params=sampling_params, request_id="1"):
    # store predicted acoustic tokens
    acoustic_tokens = output.outputs[0].custom_outputs["acoustic_tokens"]  # T x 31
    step_acoustic_tokens = acoustic_tokens[-1]  # 31,
    acoustic_tokens_lst.append(step_acoustic_tokens)

    # if previously prepared input was last, break
    if i == next_subword_ids.shape[1] - 1:
        break

    # prepare next input
    current_subword_id = next_subword_ids[:, i]  # (1,)
    next_input = embedding(
        context_text_tokens=context_subword_id.cuda(),
        audio_tokens=step_acoustic_tokens.unsqueeze(1).cuda(),
        text_tokens=current_subword_id.cuda(),
    )
    # in the next iteration use current subword id as context
    context_subword_id = current_subword_id
    new_custom_inputs = {"total_embeddings": next_input.detach().cpu().to(torch_type)}
    await engine.append_request(request_id="1", custom_inputs=new_custom_inputs)
    i += 1

acoustic_tokens_arr = torch.stack(acoustic_tokens_lst)
torch.save(acoustic_tokens_arr.detach().cpu(), "pred_tokens.pt")